# Exploration Analysis — Group 6 (Sparse / BM25 RAG), Phase 1

Statistical exploration of the frozen v1.1 release. The notebook reads only files already in the repository
(`data/metadata.json`, `data/captions_processed_v1.1.json`, `splits/test_manifest.json`) and writes tables to `exploration/`
and figures to `exploration/visualizations/`.

| Part | Content |
|---|---|
| 1 | Captions **before** and **after** preprocessing (length, vocabulary, punctuation, case) |
| 2 | Object statistics from the YOLOv8m detections: top-50 classes, objects per image, scene complexity, confidence, object size |
| 3 | Co-occurrence matrix of object classes |
| 4 | Spatial relationships computed from the bounding boxes |
| 5 | Diversity: colours, shapes and patterns, indoor/outdoor, day/night (weak labels, see the caveat in that part) |
| 6 | Scatter plot (objects per image vs caption length) |
| 7 | Part-of-speech distribution |

Run all cells from top to bottom. It needs only `numpy`, `pandas`, `matplotlib` (and `nltk` for part 7).

In [ ]:
import os, json, re, itertools, random, subprocess, sys
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})

def find_root():
    cands = []
    if os.environ.get("REPO_ROOT"):
        cands.append(Path(os.environ["REPO_ROOT"]))
    here = Path.cwd().resolve()
    cands += [here, here.parent, here.parent.parent]
    cands += [Path(p) for p in ["/content/image_processing", "/content/image_processing_project-main",
                                "/kaggle/working/image_processing_project-main"]]
    for c in cands:
        if (c / "data" / "metadata.json").exists():
            return c
    raise FileNotFoundError("Repository root not found: run this notebook from the repository (it must contain data/metadata.json) or set REPO_ROOT.")

ROOT = find_root()
OUT = ROOT / "exploration"
FIG = OUT / "visualizations"
try:
    FIG.mkdir(parents=True, exist_ok=True)
except OSError:
    OUT = Path("/tmp/exploration"); FIG = OUT / "visualizations"; FIG.mkdir(parents=True, exist_ok=True)
print("Repository root:", ROOT)
print("Outputs go to  :", OUT)

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG / name, dpi=150, bbox_inches="tight")
    plt.show()

def save_table(df, name):
    df.to_csv(OUT / name, index=False)
    return df

In [ ]:
meta = json.load(open(ROOT / "data" / "metadata.json", encoding="utf-8"))
caps = json.load(open(ROOT / "data" / "captions_processed_v1.1.json", encoding="utf-8"))
test_manifest = json.load(open(ROOT / "splits" / "test_manifest.json", encoding="utf-8"))

stratum = {x["image_id"]: x["stratum"] for x in test_manifest}
cap_by_id = {r["image_id"]: r for r in caps}
records = meta["data"]

print("Detector:", meta["detector"], "| confidence threshold:", meta["conf_threshold"])
print("Metadata records:", len(records), "| caption records:", len(caps))
print("Images per split:", dict(Counter(r["split"] for r in caps)))

## Part 1 — Captions before and after preprocessing

* **Before**: `captions_raw` of the 10,000 knowledge-base images (50,000 captions, original text, split on whitespace).
* **After**: `captions_clean` / `tokens` of the same images in release v1.1 (lowercased, punctuation removed, 223 leaking captions excluded).

Only the knowledge base is analysed here because it is the text the retriever will index.

In [ ]:
kb = [r for r in caps if r["split"] == "kb"]
raw_caps   = [c for r in kb for c in r["captions_raw"]]
clean_caps = [c for r in kb for c in r["captions_clean"]]
clean_toks = [t for r in kb for t in r["tokens"]]
raw_toks   = [c.split() for c in raw_caps]

def vocab_stats(token_lists):
    cnt = Counter(t for toks in token_lists for t in toks)
    n_tok = sum(cnt.values())
    hapax = sum(1 for v in cnt.values() if v == 1)
    return cnt, n_tok, hapax

raw_cnt, raw_ntok, raw_hapax = vocab_stats(raw_toks)
cln_cnt, cln_ntok, cln_hapax = vocab_stats(clean_toks)
raw_len = np.array([len(t) for t in raw_toks]); cln_len = np.array([len(t) for t in clean_toks])

has_upper = np.mean([any(ch.isupper() for ch in c) for c in raw_caps]) * 100
has_punct = np.mean([bool(re.search(r"[^\w\s']", c)) for c in raw_caps]) * 100
ends_punct = np.mean([c.strip()[-1:] in ".!?" for c in raw_caps]) * 100

ba = pd.DataFrame([
    ["Images", len(kb), len(kb)],
    ["Captions", len(raw_caps), len(clean_caps)],
    ["Mean words per caption", raw_len.mean().round(3), cln_len.mean().round(3)],
    ["Median words per caption", float(np.median(raw_len)), float(np.median(cln_len))],
    ["Min / max words", f"{raw_len.min()} / {raw_len.max()}", f"{cln_len.min()} / {cln_len.max()}"],
    ["Total tokens", raw_ntok, cln_ntok],
    ["Vocabulary size (distinct tokens)", len(raw_cnt), len(cln_cnt)],
    ["Hapax legomena (tokens seen once)", raw_hapax, cln_hapax],
    ["Type-token ratio", round(len(raw_cnt)/raw_ntok, 4), round(len(cln_cnt)/cln_ntok, 4)],
    ["Captions containing an uppercase letter (%)", round(has_upper, 2), 0.0],
    ["Captions containing punctuation (%)", round(has_punct, 2), 0.0],
    ["Captions ending with . ! ? (%)", round(ends_punct, 2), 0.0],
], columns=["Statistic", "Before preprocessing (raw)", "After preprocessing (v1.1 clean)"])
save_table(ba, "01_caption_before_after_preprocessing.csv")
ba

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
bins = np.arange(0, 32) - 0.5
ax[0].hist(raw_len, bins=bins, alpha=0.6, label="raw", color="#4C72B0")
ax[0].hist(cln_len, bins=bins, alpha=0.6, label="clean", color="#DD8452")
ax[0].set_xlabel("Words per caption"); ax[0].set_ylabel("Captions"); ax[0].set_title("Caption length"); ax[0].legend()
ax[1].bar(["raw", "clean"], [len(raw_cnt), len(cln_cnt)], color=["#4C72B0", "#DD8452"])
for i, v in enumerate([len(raw_cnt), len(cln_cnt)]):
    ax[1].text(i, v, f"{v:,}", ha="center", va="bottom")
ax[1].set_title("Vocabulary size"); ax[1].set_ylabel("Distinct tokens")
savefig("fig01_caption_before_after_preprocessing.png")

top_words = pd.DataFrame(cln_cnt.most_common(25), columns=["token", "count"])
save_table(top_words, "01_top25_tokens_clean.csv")
top_words.T

## Part 2 — Object statistics (from the YOLOv8m detections)

The objects are extracted directly from the images, so every statistic here starts from the detections.
Bounding boxes are in pixels (`x1, y1, x2, y2`); object size classes follow the COCO convention
(small < 32², medium < 96², large otherwise, in pixels²).

In [ ]:
rows = []
for r in records:
    for o in r["objects"]:
        x1, y1, x2, y2 = o["bbox"]
        rows.append((r["image_id"], r["split"], o["label"], o["conf"], x1, y1, x2, y2))
det = pd.DataFrame(rows, columns=["image_id", "split", "label", "conf", "x1", "y1", "x2", "y2"])
det["w"] = det.x2 - det.x1; det["h"] = det.y2 - det.y1; det["area"] = det.w * det.h
det["size"] = pd.cut(det.area, [-1, 32**2, 96**2, 1e12], labels=["small", "medium", "large"])

per_img = pd.DataFrame({"image_id": [r["image_id"] for r in records], "split": [r["split"] for r in records]})
cnt = det.groupby("image_id").size()
cls = det.groupby("image_id").label.nunique()
per_img["n_objects"] = per_img.image_id.map(cnt).fillna(0).astype(int)
per_img["n_classes"] = per_img.image_id.map(cls).fillna(0).astype(int)

obj_stats = {
    "metadata_image_records": len(records),
    "total_detected_objects": int(len(det)),
    "distinct_classes": int(det.label.nunique()),
    "mean_objects_per_image": round(float(per_img.n_objects.mean()), 3),
    "median_objects_per_image": float(per_img.n_objects.median()),
    "max_objects_in_one_image": int(per_img.n_objects.max()),
    "images_without_objects": int((per_img.n_objects == 0).sum()),
    "mean_distinct_classes_per_image": round(float(per_img.n_classes.mean()), 3),
    "mean_confidence": round(float(det.conf.mean()), 4),
    "median_confidence": round(float(det.conf.median()), 4),
}
json.dump(obj_stats, open(OUT / "02_object_statistics.json", "w"), indent=2)
pd.Series(obj_stats).to_frame("value")

In [ ]:
top50 = det.label.value_counts().head(50).rename_axis("class").reset_index(name="detections")
top50["share_%"] = (top50.detections / len(det) * 100).round(2)
top50["images_containing"] = [det[det.label == c].image_id.nunique() for c in top50["class"]]
save_table(top50, "02_top50_object_classes.csv")

fig, ax = plt.subplots(figsize=(8, 11))
ax.barh(top50["class"][::-1], top50.detections[::-1], color="#4C72B0")
ax.set_xlabel("Detections"); ax.set_title("Top-50 detected object classes (11,100 images)")
ax.set_xscale("log")
savefig("fig02_top50_object_classes.png")
top50.head(10)

In [ ]:
bins_def = [(0, 0, "0"), (1, 3, "1-3"), (4, 7, "4-7"), (8, 12, "8-12"), (13, 999, "13+")]
def bucket(n):
    for lo, hi, name in bins_def:
        if lo <= n <= hi: return name
per_img["complexity"] = per_img.n_objects.map(bucket)
order = [b[2] for b in bins_def]
comp = per_img.groupby(["split", "complexity"]).size().unstack(fill_value=0).reindex(columns=order, fill_value=0)
comp_pct = (comp.div(comp.sum(axis=1), axis=0) * 100).round(1)
save_table(comp_pct.reset_index(), "02_scene_complexity_by_split_pct.csv")

per_img["stratum"] = per_img.image_id.map(stratum)
strat = per_img[per_img.stratum.notna()].groupby("stratum").n_objects.agg(["count", "mean", "median", "min", "max"]).round(2)
save_table(strat.reset_index(), "02_objects_per_stratum_test.csv")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(per_img.n_objects.clip(upper=25), bins=np.arange(0, 27) - 0.5, color="#4C72B0")
ax[0].axvline(per_img.n_objects.mean(), color="red", ls="--", label=f"mean = {per_img.n_objects.mean():.2f}")
ax[0].set_xlabel("Objects per image (clipped at 25)"); ax[0].set_ylabel("Images"); ax[0].set_title("Scene complexity"); ax[0].legend()
data_box = [per_img[per_img.stratum == s].n_objects for s in ["simple", "medium", "complex"]]
ax[1].boxplot(data_box, showfliers=False)
ax[1].set_xticks([1, 2, 3]); ax[1].set_xticklabels(["simple", "medium", "complex"])
ax[1].set_ylabel("Objects per image"); ax[1].set_title("Test set: objects per stratum")
savefig("fig03_scene_complexity.png")
display(comp_pct); display(strat)

In [ ]:
rows = []
for t in [0.25, 0.5, 0.7, 0.9]:
    sub = det[det.conf >= t]
    n_per = sub.groupby("image_id").size().reindex(per_img.image_id).fillna(0)
    rows.append([f">= {t}", len(sub), round(float(n_per.mean()), 2), int((n_per == 0).sum()), sub.label.nunique()])
sens = pd.DataFrame(rows, columns=["Confidence filter", "Objects kept", "Mean objects per image", "Images without objects", "Classes kept"])
save_table(sens, "02_confidence_sensitivity.csv")

size_dist = det["size"].value_counts(normalize=True).mul(100).round(1).reindex(["small", "medium", "large"])
save_table(size_dist.rename_axis("object_size").reset_index(name="share_%"), "02_object_size_distribution.csv")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(det.conf, bins=30, color="#55A868")
ax[0].set_xlabel("Detection confidence"); ax[0].set_ylabel("Detections"); ax[0].set_title("Confidence distribution (threshold 0.25)")
ax[1].bar(size_dist.index.astype(str), size_dist.values, color="#C44E52")
ax[1].set_ylabel("% of detections"); ax[1].set_title("Object size (COCO convention)")
savefig("fig04_confidence_and_size.png")
sens

## Part 3 — Co-occurrence of object classes

Number of images in which two classes are both detected (top-20 classes by number of detections).

In [ ]:
top20 = list(det.label.value_counts().head(20).index)
img_sets = det.groupby("image_id").label.agg(set)
M = pd.DataFrame(0, index=top20, columns=top20)
for s in img_sets:
    present = [c for c in top20 if c in s]
    for a, b in itertools.combinations(present, 2):
        M.loc[a, b] += 1; M.loc[b, a] += 1
M.to_csv(OUT / "03_cooccurrence_top20.csv")

pairs = [(a, b, int(M.loc[a, b])) for a, b in itertools.combinations(top20, 2)]
pairs = pd.DataFrame(sorted(pairs, key=lambda x: -x[2])[:15], columns=["class_a", "class_b", "images_with_both"])
save_table(pairs, "03_top15_cooccurring_pairs.csv")

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(np.log1p(M.values), cmap="viridis")
ax.set_xticks(range(20)); ax.set_xticklabels(top20, rotation=90); ax.set_yticks(range(20)); ax.set_yticklabels(top20)
ax.grid(False); ax.set_title("Co-occurrence of the 20 most frequent classes (colour = log(1 + images))")
plt.colorbar(im, ax=ax, shrink=0.8)
savefig("fig05_cooccurrence_matrix.png")
pairs

## Part 4 — Spatial relationships (from bounding boxes)

For every pair of detections in an image (detections with confidence >= 0.5, at most the 12 most confident objects per image),
the relation of object A with respect to object B is:

* `inside` / `contains`: at least 80 % of the smaller box lies in the larger one,
* `overlaps`: the boxes intersect (IoU >= 0.05),
* otherwise, along the dominant axis of the displacement between box centres: `above`, `below`, `left_of`, `right_of`.

The pair is ordered alphabetically by class label so each pair is counted once. For two objects of the **same** class the direction is arbitrary, so their relation is made symmetric (`beside`, `stacked`, `nested`, `overlaps`) and reported separately.

In [ ]:
def relation(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    ix = max(0.0, min(ax2, bx2) - max(ax1, bx1)); iy = max(0.0, min(ay2, by2) - max(ay1, by1))
    inter = ix * iy
    area_a = max((ax2 - ax1) * (ay2 - ay1), 1e-6); area_b = max((bx2 - bx1) * (by2 - by1), 1e-6)
    if inter / min(area_a, area_b) >= 0.8:
        return "inside" if area_a <= area_b else "contains"
    if inter > 0 and inter / (area_a + area_b - inter) >= 0.05:
        return "overlaps"
    dx = (ax1 + ax2) / 2 - (bx1 + bx2) / 2; dy = (ay1 + ay2) / 2 - (by1 + by2) / 2
    if abs(dy) > abs(dx):
        return "above" if dy < 0 else "below"
    return "left_of" if dx < 0 else "right_of"

SYMMETRIC = {"left_of": "beside", "right_of": "beside", "above": "stacked", "below": "stacked", "inside": "nested", "contains": "nested"}
rel_counter = {"different_class": Counter(), "same_class": Counter()}
trip_counter = Counter()
pairs_per_image = []
for r in records:
    objs = sorted([o for o in r["objects"] if o["conf"] >= 0.5], key=lambda o: -o["conf"])[:12]
    n_pairs = 0
    for o1, o2 in itertools.combinations(objs, 2):
        if (o1["label"], o1["bbox"][0]) > (o2["label"], o2["bbox"][0]):
            o1, o2 = o2, o1
        rel = relation(o1["bbox"], o2["bbox"])
        if o1["label"] == o2["label"]:
            rel = SYMMETRIC.get(rel, rel)          # same class: direction is arbitrary, keep symmetric relations
            rel_counter["same_class"][rel] += 1
        else:
            rel_counter["different_class"][rel] += 1
        trip_counter[(o1["label"], rel, o2["label"])] += 1
        n_pairs += 1
    pairs_per_image.append(n_pairs)

rel_df = pd.DataFrame([(fam, rel, n) for fam, c in rel_counter.items() for rel, n in c.most_common()],
                      columns=["pair_type", "relation", "pairs"])
rel_df["share_within_type_%"] = (rel_df.pairs / rel_df.groupby("pair_type").pairs.transform("sum") * 100).round(2)
save_table(rel_df, "04_spatial_relations_distribution.csv")
trip_df = pd.DataFrame([(a, r, b, n) for (a, r, b), n in trip_counter.most_common(30)], columns=["object_a", "relation", "object_b", "pairs"])
save_table(trip_df, "04_spatial_triplets_top30.csv")

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
for a_, fam, col in [(ax[0], "different_class", "#8172B3"), (ax[1], "same_class", "#64B5CD")]:
    sub = rel_df[rel_df.pair_type == fam]
    a_.bar(sub.relation, sub.pairs, color=col); a_.set_title(f"Relations, {fam.replace('_', '-')} pairs"); a_.set_ylabel("Pairs"); a_.tick_params(axis="x", rotation=30)
ax[2].hist(pd.Series(pairs_per_image).clip(upper=40), bins=40, color="#C44E52")
ax[2].set_xlabel("Object pairs per image (clipped at 40)"); ax[2].set_ylabel("Images"); ax[2].set_title("Relational complexity")
savefig("fig06_spatial_relations.png")
display(rel_df); display(trip_df.head(10))

## Part 5 — Diversity analysis (colours, shapes, context)

The images themselves are not stored in the repository, so these attributes are estimated from the cleaned captions
(and, for indoor/outdoor, from the detected object classes). They are **weak labels**: an image is counted only when
its five captions explicitly mention the attribute, so the percentages are lower bounds on how often the attribute is
really present. They describe the diversity of the data, not ground truth.

In [ ]:
img_text = {r["image_id"]: set(" ".join(r["captions_clean"]).split()) for r in caps}
img_split = {r["image_id"]: r["split"] for r in caps}
N = len(img_text)

def share(words, merge=None):
    merge = merge or {}
    c = Counter()
    for toks in img_text.values():
        found = {merge.get(w, w) for w in words if w in toks}
        for w in found: c[w] += 1
    s = pd.Series(c).sort_values(ascending=False)
    return (s / N * 100).round(2)

COLORS = ["red","orange","yellow","green","blue","purple","pink","brown","black","white","gray","grey","silver","gold","golden","tan","beige","turquoise"]
SHAPES = ["round","circular","square","rectangular","oval","triangular","long","tall","flat","curved"]
PATTERNS = ["striped","spotted","checkered","plaid","patterned","floral"]

col_s = share(COLORS, merge={"grey": "gray", "golden": "gold"})
shape_s = share(SHAPES)
pat_s = share(PATTERNS)
attr = pd.concat([pd.DataFrame({"attribute": col_s.index, "type": "colour", "images_%": col_s.values}),
                  pd.DataFrame({"attribute": shape_s.index, "type": "shape", "images_%": shape_s.values}),
                  pd.DataFrame({"attribute": pat_s.index, "type": "pattern", "images_%": pat_s.values})])
save_table(attr, "05_attribute_diversity.csv")

any_color = np.mean([any(w in t for w in COLORS) for t in img_text.values()]) * 100
print(f"Images whose captions mention at least one colour: {any_color:.1f}%")

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].bar(col_s.index, col_s.values, color="#4C72B0"); ax[0].set_title("Colour words (% of images)"); ax[0].tick_params(axis="x", rotation=60)
ax[1].bar(shape_s.index, shape_s.values, color="#55A868"); ax[1].set_title("Shape words (% of images)"); ax[1].tick_params(axis="x", rotation=60)
ax[2].bar(pat_s.index, pat_s.values, color="#C44E52"); ax[2].set_title("Pattern words (% of images)"); ax[2].tick_params(axis="x", rotation=60)
savefig("fig07_attribute_diversity.png")
attr.head(12)

In [ ]:
INDOOR_W  = {"kitchen","bathroom","bedroom","room","indoor","indoors","inside","living","office","restaurant","table","bed","couch","sofa","toilet","sink","counter","desk","shelf","wall","oven","fridge","refrigerator"}
OUTDOOR_W = {"street","outside","outdoor","outdoors","field","beach","snow","park","road","sky","grass","mountain","ocean","water","city","sidewalk","airport","farm","hill","lake","forest","river","track","runway","court","slope","wave","waves"}
INDOOR_O  = {"bed","couch","tv","refrigerator","oven","microwave","toaster","sink","toilet","laptop","keyboard","remote","book","clock","vase","bowl","dining table","cup","fork","knife","spoon","teddy bear","hair drier","toothbrush"}
OUTDOOR_O = {"traffic light","car","bus","truck","train","airplane","boat","kite","surfboard","skis","snowboard","fire hydrant","stop sign","parking meter","bench","bicycle","motorcycle","giraffe","elephant","zebra","horse","cow","sheep","bird","umbrella","sports ball","frisbee","skateboard","baseball bat","baseball glove","tennis racket"}
NIGHT_W = {"night","nighttime","dark","evening","dusk","moonlit","midnight"}
DAY_W   = {"sunny","daytime","sunlight","sunshine","noon","morning","afternoon","day"}

labels_by_img = det.groupby("image_id").label.agg(lambda s: set(s))
ctx, tod = {}, {}
for iid, toks in img_text.items():
    objs = labels_by_img.get(iid, set())
    ind = len(toks & INDOOR_W) + len(objs & INDOOR_O)
    out = len(toks & OUTDOOR_W) + len(objs & OUTDOOR_O)
    ctx[iid] = "indoor" if ind > out else "outdoor" if out > ind else "unclear"
    nt, dy = len(toks & NIGHT_W), len(toks & DAY_W)
    tod[iid] = "night" if nt > dy else "day" if dy > nt else "not stated"

ctx_df = pd.crosstab(pd.Series(img_split), pd.Series(ctx), normalize="index").mul(100).round(1)
tod_df = pd.crosstab(pd.Series(img_split), pd.Series(tod), normalize="index").mul(100).round(1)
save_table(ctx_df.reset_index(), "05_indoor_outdoor_by_split_pct.csv")
save_table(tod_df.reset_index(), "05_day_night_by_split_pct.csv")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ctx_df[["indoor", "outdoor", "unclear"]].plot(kind="bar", stacked=True, ax=ax[0], color=["#4C72B0", "#55A868", "#BBBBBB"], rot=0)
ax[0].set_ylabel("% of images"); ax[0].set_title("Indoor / outdoor (weak labels)")
tod_df[["day", "night", "not stated"]].plot(kind="bar", stacked=True, ax=ax[1], color=["#F2C14E", "#2F3E5C", "#BBBBBB"], rot=0)
ax[1].set_ylabel("% of images"); ax[1].set_title("Day / night (explicitly mentioned only)")
savefig("fig08_context_diversity.png")
display(ctx_df); display(tod_df)

## Part 6 — Scatter plot: objects per image vs caption length

Does a more complex scene (more detected objects) produce longer captions? One point per knowledge-base image.

In [ ]:
mean_len = {r["image_id"]: np.mean([len(t) for t in r["tokens"]]) for r in kb}
sc = per_img[per_img.split == "kb"].copy()
sc["mean_caption_len"] = sc.image_id.map(mean_len)
corr = sc[["n_objects", "mean_caption_len"]].corr().iloc[0, 1]
slope, intercept = np.polyfit(sc.n_objects, sc.mean_caption_len, 1)

fig, ax = plt.subplots(figsize=(7, 5))
jitter = np.random.uniform(-0.2, 0.2, len(sc))
ax.scatter(sc.n_objects + jitter, sc.mean_caption_len, s=6, alpha=0.25, color="#4C72B0")
xs = np.linspace(0, sc.n_objects.max(), 50)
ax.plot(xs, slope * xs + intercept, color="red", label=f"linear fit (Pearson r = {corr:.3f})")
ax.set_xlabel("Detected objects per image"); ax.set_ylabel("Mean caption length (words)")
ax.set_title("Scene complexity vs caption length (kb images)"); ax.legend()
savefig("fig09_objects_vs_caption_length.png")
save_table(pd.DataFrame([{"pearson_r": round(corr, 4), "slope_words_per_object": round(slope, 4), "intercept": round(intercept, 3), "images": len(sc)}]), "06_scatter_correlation.csv")

## Part 7 — Part-of-speech distribution

NLTK tagger on a random sample of 10,000 cleaned knowledge-base captions (seed 42). Penn Treebank tags are grouped into
coarse classes. The tagger is downloaded the first time (internet needed once).

In [ ]:
def coarse(tag):
    if tag.startswith("NN"): return "Noun"
    if tag.startswith("VB"): return "Verb"
    if tag.startswith("JJ"): return "Adjective"
    if tag.startswith("RB"): return "Adverb"
    if tag in ("DT", "PDT", "WDT"): return "Determiner"
    if tag in ("IN", "TO"): return "Preposition"
    if tag == "CC": return "Conjunction"
    if tag == "CD": return "Numeral"
    if tag.startswith("PRP") or tag in ("WP", "WP$"): return "Pronoun"
    return "Other"

try:
    try:
        import nltk
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nltk"], check=True)
        import nltk
    for pkg in ["averaged_perceptron_tagger_eng", "averaged_perceptron_tagger"]:
        try: nltk.download(pkg, quiet=True)
        except Exception: pass
    rng = random.Random(SEED)
    sample = rng.sample(clean_toks, min(10000, len(clean_toks)))
    counts = Counter(coarse(t) for toks in sample for _, t in nltk.pos_tag(toks))
    pos_df = pd.DataFrame(counts.most_common(), columns=["pos_class", "tokens"])
    pos_df["share_%"] = (pos_df.tokens / pos_df.tokens.sum() * 100).round(2)
    save_table(pos_df, "07_pos_distribution.csv")
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(pos_df.pos_class, pos_df["share_%"], color="#8172B3"); ax.set_ylabel("% of tokens")
    ax.set_title("Part-of-speech distribution (10,000 captions)"); ax.tick_params(axis="x", rotation=30)
    savefig("fig10_pos_distribution.png")
    display(pos_df)
except Exception as e:
    print("POS tagging skipped:", type(e).__name__, e)
    print("Fix: run  pip install nltk  then  python -c \"import nltk; nltk.download('averaged_perceptron_tagger_eng')\"  and re-run this cell.")

## Summary

In [ ]:
summary = {
    "release": "v1.1",
    "captions_before_preprocessing": len(raw_caps), "captions_after_preprocessing": len(clean_caps),
    "vocabulary_raw": len(raw_cnt), "vocabulary_clean": len(cln_cnt),
    "mean_words_raw": round(float(raw_len.mean()), 3), "mean_words_clean": round(float(cln_len.mean()), 3),
    **obj_stats,
    "top_class": top50.iloc[0]["class"], "top_class_detections": int(top50.iloc[0].detections),
    "spatial_pairs_analysed": int(rel_df.pairs.sum()),
    "pearson_objects_vs_caption_length": round(float(corr), 4),
}
json.dump(summary, open(OUT / "exploration_summary.json", "w"), indent=2)

print("Files written to", OUT)
for p in sorted(OUT.glob("*.*")): print("  ", p.name)
for p in sorted(FIG.glob("fig*.png")): print("   visualizations/" + p.name)
pd.Series(summary).to_frame("value")